In [1]:
# ============================================================
# D8 — Stage 4 Validation — Branch C (Structural Conversion)
# ============================================================

from google.colab import files
from pathlib import Path
from datetime import datetime
from collections import Counter
from difflib import SequenceMatcher

import hashlib
import json
import math
import platform
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment


In [2]:
# ============================================================
# 0. Configuration
# ============================================================

DOCUMENT_ID = "D8"
DOCUMENT_NAME = (
    "World Bank — Bhutan - Land Management Project — "
    "Project Information Document (PID), Concept Stage"
)

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
INPUT_REPRESENTATION = (
    "Complete deterministically normalised page-aware structural Markdown"
)

EXPECTED_RECORD_COUNT = 49

EXPECTED_CATEGORY_COUNTS = {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5,
}

FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location",
]

# Frozen from final D8 Branch A validation.
# Description remains diagnostic rather than a primary correctness field
# because Stage 1 defines it as a concise source-grounded description,
# not a unique lexical label.
PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Topic",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location",
]

DESCRIPTION_DIAGNOSTIC_FIELD = "Description"

MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location",
]

NULLABLE_STRING_FIELDS = [
    "Unit",
    "Qualifier",
    "Reporting Period",
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

EXPECTED_SOURCE_SHA256 = (
    "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5"
)

# Frozen D8 Branch A alignment design.
BLOCK_FIELDS = [
    "Category",
    "Source Location",
]

MATCHING_WEIGHTS = {
    "topic": 0.65,
    "description": 0.25,
    "reporting_period": 0.10,
}

MATCH_SCORE_THRESHOLD = 0.35

OUTPUT_DIR = Path("outputs_D8_validation_C_revised")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Fields:", len(FIELDS))
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)
print("Output directory:", OUTPUT_DIR)


Document: D8
Branch: C
Expected records: 49
Fields: 8
Primary correctness fields: ['Category', 'Topic', 'Value', 'Unit', 'Qualifier', 'Reporting Period', 'Source Location']
Output directory: outputs_D8_validation_C_revised


In [3]:
# ============================================================
# 1. Upload canonical Stage 1 + Branch C validation inputs
# ============================================================
# Required:
#   1) D8_reference_values.csv
#   2) D8_branch_C_parsed_extraction.json
#   3) D8_branch_C_structure_check.json
#   4) D8_branch_C_experiment_metadata.json
#   5) D8_branch_C_normalisation_check.json
#
# Optional:
#   6) D8_branch_C_experiment_summary.json
#
# File identification uses canonical filename patterns first.
# Content-based fallback is used only when necessary so that the
# experiment summary cannot be mistaken for the structure check.

print(
    "Upload:\n"
    "1. D8_reference_values.csv\n"
    "2. D8_branch_C_parsed_extraction.json\n"
    "3. D8_branch_C_structure_check.json\n"
    "4. D8_branch_C_experiment_metadata.json\n"
    "5. D8_branch_C_normalisation_check.json\n"
    "6. Optional: D8_branch_C_experiment_summary.json"
)

uploaded = files.upload()

uploaded_paths = [
    Path(name)
    for name in uploaded
]

csv_paths = [
    path
    for path in uploaded_paths
    if path.suffix.lower() == ".csv"
]

json_paths = [
    path
    for path in uploaded_paths
    if path.suffix.lower() == ".json"
]

if len(csv_paths) != 1:
    raise ValueError(
        "Upload exactly one CSV file: D8_reference_values.csv."
    )

REFERENCE_PATH = csv_paths[0]

EXTRACTION_PATH = None
STRUCTURE_CHECK_PATH = None
EXPERIMENT_METADATA_PATH = None
NORMALISATION_INTEGRITY_PATH = None
EXPERIMENT_SUMMARY_PATH = None


def canonical_filename(path):
    return (
        path.name
        .casefold()
        .replace(" ", "_")
    )


# ------------------------------------------------------------
# First pass: canonical filename patterns
# ------------------------------------------------------------

for path in json_paths:

    filename = canonical_filename(path)

    if "d8_branch_c_parsed_extraction" in filename:
        EXTRACTION_PATH = path
        continue

    if "d8_branch_c_structure_check" in filename:
        STRUCTURE_CHECK_PATH = path
        continue

    if "d8_branch_c_experiment_metadata" in filename:
        EXPERIMENT_METADATA_PATH = path
        continue

    if (
        "d8_branch_c_normalisation_check" in filename
        or "d8_branch_c_normalization_check" in filename
    ):
        NORMALISATION_INTEGRITY_PATH = path
        continue

    if "d8_branch_c_experiment_summary" in filename:
        EXPERIMENT_SUMMARY_PATH = path
        continue


# ------------------------------------------------------------
# Second pass: content-based fallback
# ------------------------------------------------------------

for path in json_paths:

    with path.open("r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        EXTRACTION_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        EXTRACTION_PATH = path
        continue

    if (
        EXPERIMENT_METADATA_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "parsed_extraction_sha256" in obj
        and "source_sha256" in obj
        and "structure_check_file" in obj
    ):
        EXPERIMENT_METADATA_PATH = path
        continue

    if (
        NORMALISATION_INTEGRITY_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and obj.get("parent_branch") == "B"
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_INTEGRITY_PATH = path
        continue

    # Summary must be identified before the structure-check fallback,
    # because both may contain overlapping structure diagnostics.
    if (
        EXPERIMENT_SUMMARY_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "validation_status" in obj
        and "accuracy_validation_completed" in obj
    ):
        EXPERIMENT_SUMMARY_PATH = path
        continue

    if (
        STRUCTURE_CHECK_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structure_valid" in obj
        and "record_schema_valid" in obj
        and "field_types_valid" in obj
        and "validation_status" not in obj
        and "accuracy_validation_completed" not in obj
        and "parsed_extraction_sha256" not in obj
    ):
        STRUCTURE_CHECK_PATH = path
        continue


if EXTRACTION_PATH is None:
    raise ValueError(
        "Could not identify D8_branch_C_parsed_extraction.json."
    )

if STRUCTURE_CHECK_PATH is None:
    raise ValueError(
        "Could not identify D8_branch_C_structure_check.json."
    )

if EXPERIMENT_METADATA_PATH is None:
    raise ValueError(
        "Could not identify D8_branch_C_experiment_metadata.json."
    )

if NORMALISATION_INTEGRITY_PATH is None:
    raise ValueError(
        "Could not identify D8_branch_C_normalisation_check.json."
    )


required_paths = {
    "parsed_extraction": EXTRACTION_PATH,
    "structure_check": STRUCTURE_CHECK_PATH,
    "experiment_metadata": EXPERIMENT_METADATA_PATH,
    "normalisation_check": NORMALISATION_INTEGRITY_PATH,
}

required_path_strings = [
    str(path)
    for path in required_paths.values()
]

if len(required_path_strings) != len(set(required_path_strings)):
    raise ValueError(
        "The same JSON file was assigned to more than one "
        "required Branch C artefact type."
    )


print("\nIdentified D8 Branch C validation inputs:")
print("Reference:", REFERENCE_PATH.name)
print("Parsed extraction:", EXTRACTION_PATH.name)
print("Structure check:", STRUCTURE_CHECK_PATH.name)
print("Experiment metadata:", EXPERIMENT_METADATA_PATH.name)
print("Normalisation integrity:", NORMALISATION_INTEGRITY_PATH.name)
print(
    "Experiment summary:",
    (
        EXPERIMENT_SUMMARY_PATH.name
        if EXPERIMENT_SUMMARY_PATH is not None
        else "Not supplied"
    )
)


Upload:
1. D8_reference_values.csv
2. D8_branch_C_parsed_extraction.json
3. D8_branch_C_structure_check.json
4. D8_branch_C_experiment_metadata.json
5. D8_branch_C_normalisation_check.json
6. Optional: D8_branch_C_experiment_summary.json


Saving D8_branch_C_structure_check.json to D8_branch_C_structure_check.json
Saving D8_branch_C_parsed_extraction.json to D8_branch_C_parsed_extraction.json
Saving D8_branch_C_normalisation_check.json to D8_branch_C_normalisation_check.json
Saving D8_branch_C_experiment_summary.json to D8_branch_C_experiment_summary.json
Saving D8_branch_C_experiment_metadata.json to D8_branch_C_experiment_metadata.json
Saving D8_reference_values.csv to D8_reference_values.csv

Identified D8 Branch C validation inputs:
Reference: D8_reference_values.csv
Parsed extraction: D8_branch_C_parsed_extraction.json
Structure check: D8_branch_C_structure_check.json
Experiment metadata: D8_branch_C_experiment_metadata.json
Normalisation integrity: D8_branch_C_normalisation_check.json
Experiment summary: D8_branch_C_experiment_summary.json


In [4]:
# ============================================================
# 2. File hashing utility and input hashes
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
STRUCTURE_CHECK_SHA256 = sha256_file(STRUCTURE_CHECK_PATH)
EXPERIMENT_METADATA_SHA256 = sha256_file(EXPERIMENT_METADATA_PATH)
NORMALISATION_INTEGRITY_SHA256 = sha256_file(
    NORMALISATION_INTEGRITY_PATH
)

EXPERIMENT_SUMMARY_SHA256 = (
    sha256_file(EXPERIMENT_SUMMARY_PATH)
    if EXPERIMENT_SUMMARY_PATH is not None
    else None
)

print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print("Structure-check SHA-256:", STRUCTURE_CHECK_SHA256)
print("Experiment-metadata SHA-256:", EXPERIMENT_METADATA_SHA256)
print(
    "Normalisation-integrity SHA-256:",
    NORMALISATION_INTEGRITY_SHA256
)


Reference SHA-256: f633a629a88a998c9d9e2c697d07d57a9d29c965c629a2e0b6e26c2608894514
Extraction SHA-256: cfece398b57b2278cfb5108d00ae1f7e741ff9b03e0d4cd097136f71b96e04c4
Structure-check SHA-256: 2a3a031e89432f1285d28f393a0b004f49cedcffd48da000fb14e1020a654d08
Experiment-metadata SHA-256: da903748ae6839ac8c0c208e91bd1326c05bce052c4ec36e61d4b34616ad4cac
Normalisation-integrity SHA-256: 15f8af4c3d11cbf63bcf9e66e655fe0595862c2054ec2de415a774e272f1ba0d


In [5]:
# ============================================================
# 3. Load fixed Stage 1 reference dataset
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=False,
)

# Convert empty CSV cells back to None.
reference_df = reference_df.replace("", None)

# Restore numeric Values where Stage 1 stored JSON numbers.
def restore_reference_value(value):
    if value is None:
        return None

    text = str(value).strip()

    # Preserve explicit textual quantities / codes / dates / strings.
    if text.casefold() in {"one-quarter", "one-third"}:
        return text

    # Numeric strings only when they are unambiguously numeric.
    if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
        try:
            number = float(text)
            return int(number) if number.is_integer() else number
        except ValueError:
            pass

    return value


reference_df["Value"] = reference_df["Value"].map(
    restore_reference_value
)

reference_records = reference_df.to_dict(orient="records")

reference_schema_exact = (
    reference_df.columns.tolist() == FIELDS
)

print("Reference records:", len(reference_df))
print("Reference columns:", reference_df.columns.tolist())
print("Reference schema exact:", reference_schema_exact)

display(reference_df.head(12))


Reference records: 49
Reference columns: ['Category', 'Topic', 'Description', 'Value', 'Unit', 'Qualifier', 'Reporting Period', 'Source Location']
Reference schema exact: True


,Category,Topic,Description,Value,Unit,Qualifier,Reporting Period,Source Location
0,Project metadata,Report No.,Project Information Document report number,AB526,None,None,None,DOC page 1 — Header
1,Project metadata,Project Name,Project name,Bhutan - Land Management Project,None,None,None,DOC page 1 — Header
2,Project metadata,Region,World Bank region,SOUTH ASIA,None,None,None,DOC page 1 — Header
3,Project metadata,Sector,Represented sector allocation,"General agriculture, fishing and forestry sect...",None,None,None,DOC page 1 — Header
4,Project metadata,Project ID,World Bank project identifier,P087039,None,None,None,DOC page 1 — Header
5,Project metadata,GEF Focal Area,Global Environment Facility focal area,L-Land degradation,None,None,None,DOC page 1 — Header
6,Project metadata,Borrower(s),Project borrower,RGOB,None,None,None,DOC page 1 — Header
7,Project metadata,Implementing Agency,Represented implementing-agency status and lik...,TBD (Most likely Ministry of Agriculture and/o...,None,None,None,DOC page 1 — Header
8,Project metadata,Environment Category,Selected environment category,B,category,None,None,DOC page 1 — Header
9,Project metadata,Safeguard Classification,Selected safeguard classification,S2,classification,None,None,DOC page 1 — Header


In [6]:
# ============================================================
# 4. Load canonical preserved Branch C extraction
# ============================================================

with EXTRACTION_PATH.open("r", encoding="utf-8") as f:
    extraction_content = json.load(f)

valid_json = True
top_level_object_valid = isinstance(extraction_content, dict)

document_id_correct = (
    top_level_object_valid
    and extraction_content.get("document_id") == DOCUMENT_ID
)

branch_correct = (
    top_level_object_valid
    and extraction_content.get("branch") == BRANCH
)

records_is_list = (
    top_level_object_valid
    and isinstance(extraction_content.get("records"), list)
)

if not records_is_list:
    raise ValueError(
        "Use the canonical D8_branch_C_parsed_extraction.json "
        "generated by the current Branch C notebook."
    )

extracted_records = extraction_content["records"]

print("Document ID correct:", document_id_correct)
print("Branch correct:", branch_correct)
print("Extracted records:", len(extracted_records))


Document ID correct: True
Branch correct: True
Extracted records: 49


In [7]:
# ============================================================
# 5. Load Branch C structure check and experiment metadata
# ============================================================

with STRUCTURE_CHECK_PATH.open(
    "r",
    encoding="utf-8"
) as f:
    structure_check = json.load(f)

with EXPERIMENT_METADATA_PATH.open(
    "r",
    encoding="utf-8"
) as f:
    experiment_metadata = json.load(f)


branch_c_structure_valid = bool(
    structure_check.get("structure_valid")
)

metadata_document_id_correct = (
    experiment_metadata.get("document_id")
    == DOCUMENT_ID
)

metadata_branch_correct = (
    experiment_metadata.get("branch")
    == BRANCH
)

metadata_parsed_hash = (
    experiment_metadata.get(
        "parsed_extraction_sha256"
    )
)

parsed_extraction_hash_matches_metadata = (
    metadata_parsed_hash
    == EXTRACTION_SHA256
)

metadata_source_hash = (
    experiment_metadata.get(
        "source_sha256"
    )
)

source_hash_matches_stage_1 = (
    metadata_source_hash
    == EXPECTED_SOURCE_SHA256
)


print("Branch C structure valid:", branch_c_structure_valid)
print(
    "Metadata document ID correct:",
    metadata_document_id_correct
)
print(
    "Metadata branch correct:",
    metadata_branch_correct
)
print(
    "Parsed extraction hash matches Branch C metadata:",
    parsed_extraction_hash_matches_metadata
)
print(
    "Source hash matches fixed Stage 1 source:",
    source_hash_matches_stage_1
)


if not metadata_document_id_correct:
    raise AssertionError(
        "Branch C experiment metadata has the wrong document_id."
    )

if not metadata_branch_correct:
    raise AssertionError(
        "Branch C experiment metadata has the wrong branch."
    )

if not parsed_extraction_hash_matches_metadata:
    raise AssertionError(
        "The uploaded parsed extraction is not the canonical "
        "Branch C extraction recorded in the experiment metadata."
    )

if not source_hash_matches_stage_1:
    raise AssertionError(
        "The Branch C source hash does not match the fixed "
        "D8 Stage 1 source."
    )


Branch C structure valid: True
Metadata document ID correct: True
Metadata branch correct: True
Parsed extraction hash matches Branch C metadata: True
Source hash matches fixed Stage 1 source: True


In [8]:
# ============================================================
# 6. Assess extraction record schema
# ============================================================

schema_issue_rows = []
field_order_diagnostic_rows = []

for record_index, record in enumerate(extracted_records):

    if not isinstance(record, dict):
        schema_issue_rows.append({
            "Record Index": record_index,
            "Issue": "Record is not a JSON object",
        })
        continue

    observed_fields = list(record.keys())
    observed_field_set = set(observed_fields)
    expected_field_set = set(FIELDS)

    missing_fields = [
        field for field in FIELDS
        if field not in observed_field_set
    ]

    extra_fields = [
        field for field in observed_fields
        if field not in expected_field_set
    ]

    if missing_fields or extra_fields:
        schema_issue_rows.append({
            "Record Index": record_index,
            "Issue": "Field set mismatch",
            "Missing Fields": ", ".join(missing_fields),
            "Extra Fields": ", ".join(extra_fields),
        })

    # JSON object order is not semantic schema validity.
    if observed_fields != FIELDS:
        field_order_diagnostic_rows.append({
            "Record Index": record_index,
            "Observed Order": observed_fields,
            "Expected Order": FIELDS,
        })


schema_issues_df = pd.DataFrame(schema_issue_rows)
field_order_diagnostics_df = pd.DataFrame(
    field_order_diagnostic_rows
)

record_schema_valid = schema_issues_df.empty

print("Record schema valid:", record_schema_valid)
print("Schema issues:", len(schema_issues_df))
print(
    "Field-order diagnostics:",
    len(field_order_diagnostics_df),
)

if not schema_issues_df.empty:
    display(schema_issues_df)


Record schema valid: True
Schema issues: 0
Field-order diagnostics: 0


In [9]:
# ============================================================
# 7. Reference integrity and extraction content diagnostics
# ============================================================

reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts == EXPECTED_CATEGORY_COUNTS
)

extracted_record_count = len(extracted_records)

extraction_record_count_valid = (
    extracted_record_count == EXPECTED_RECORD_COUNT
)

extraction_category_counts = dict(
    Counter(
        record.get("Category")
        for record in extracted_records
        if isinstance(record, dict)
    )
)

extraction_category_counts_valid = (
    extraction_category_counts == EXPECTED_CATEGORY_COUNTS
)

print("Reference record count valid:", reference_record_count_valid)
print("Reference category counts valid:", reference_category_counts_valid)
print("Extraction record count valid:", extraction_record_count_valid)
print("Extraction category counts valid:", extraction_category_counts_valid)

if not reference_record_count_valid:
    raise AssertionError(
        "The fixed Stage 1 reference dataset does not contain 49 records."
    )

if not reference_category_counts_valid:
    raise AssertionError(
        "The fixed Stage 1 reference category distribution is invalid."
    )


Reference record count valid: True
Reference category counts valid: True
Extraction record count valid: True
Extraction category counts valid: True


In [10]:
# ============================================================
# 8. Null-safe type and mandatory-content diagnostics
# ============================================================

type_issue_rows = []
missing_mandatory_rows = []

for record_index, record in enumerate(extracted_records):

    if not isinstance(record, dict):
        continue

    for field in MANDATORY_STRING_FIELDS:
        value = record.get(field)

        if value is None or value == "":
            missing_mandatory_rows.append({
                "Record Index": record_index,
                "Field": field,
            })
        elif not isinstance(value, str):
            type_issue_rows.append({
                "Record Index": record_index,
                "Field": field,
                "Observed Type": type(value).__name__,
                "Expected Type": "string",
            })

    for field in NULLABLE_STRING_FIELDS:
        value = record.get(field)

        if value is not None and not isinstance(value, str):
            type_issue_rows.append({
                "Record Index": record_index,
                "Field": field,
                "Observed Type": type(value).__name__,
                "Expected Type": "string or null",
            })

    value = record.get("Value")

    if (
        value is not None
        and (
            isinstance(value, bool)
            or not isinstance(value, (str, int, float))
        )
    ):
        type_issue_rows.append({
            "Record Index": record_index,
            "Field": "Value",
            "Observed Type": type(value).__name__,
            "Expected Type": "string, number or null",
        })


type_issues_df = pd.DataFrame(type_issue_rows)
missing_mandatory_fields_df = pd.DataFrame(
    missing_mandatory_rows
)

field_types_valid = type_issues_df.empty
mandatory_fields_complete = missing_mandatory_fields_df.empty

print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)


Field types valid: True
Mandatory fields complete: True


In [11]:
# ============================================================
# 9. Comparison-only text normalisation and similarity
# ============================================================

def is_missing(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def normalise_text(value):
    if is_missing(value):
        return None

    text = str(value)

    text = "".join(
        character
        for character in text
        if unicodedata.category(character) != "Cf"
    )

    text = unicodedata.normalize("NFKC", text)

    text = (
        text.replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00a0", " ")
    )

    text = re.sub(r"\s+", " ", text).strip()
    return text.casefold()


def identity_text(value):
    text = normalise_text(value)

    if text is None:
        return ""

    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def text_similarity(first, second):
    first_text = identity_text(first)
    second_text = identity_text(second)

    if not first_text and not second_text:
        return 1.0

    if not first_text or not second_text:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text,
    ).ratio()


def exact_normalised_text_match(first, second):
    return normalise_text(first) == normalise_text(second)


In [12]:
# ============================================================
# 10. Conservative D8 comparison equivalence rules
# ============================================================

# FROZEN FROM FINAL D8 BRANCH A VALIDATION — REUSED UNCHANGED.
# First-run policy:
# - Do NOT create Topic aliases before discrepancy inspection.
# - Do NOT forgive repaired source typos/codes.
# - Do NOT move qualifier wording into Unit.
# - Do NOT use fuzzy similarity for correctness.
# - Only safe representation equivalence is applied initially.

UNIT_EQUIVALENCE_MAP = {
    "%": "percent",
    "percent": "percent",
    "usd million": "usd million",
    "million usd": "usd million",
    "$ million": "usd million",
    "usd millions": "usd million",
}


def canonical_unit(value):
    text = normalise_text(value)

    if text is None:
        return None

    return UNIT_EQUIVALENCE_MAP.get(text, text)


def canonical_period(value):
    return normalise_text(value)


def canonical_qualifier(value):
    return normalise_text(value)


def canonical_topic(value):
    # Intentionally conservative for the first D8 run.
    return normalise_text(value)


def topic_correct(reference_value, extracted_value):
    return (
        canonical_topic(reference_value)
        == canonical_topic(extracted_value)
    )


def unit_correct(reference_value, extracted_value):
    return (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    )


def qualifier_correct(reference_value, extracted_value):
    return (
        canonical_qualifier(reference_value)
        == canonical_qualifier(extracted_value)
    )


def period_correct(reference_value, extracted_value):
    return (
        canonical_period(reference_value)
        == canonical_period(extracted_value)
    )


print(
    "Conservative D8 rules loaded. "
    "No Branch-A-specific Topic aliases are active."
)


Conservative D8 rules loaded. No Branch-A-specific Topic aliases are active.


In [13]:
# ============================================================
# 11. Null-safe Value comparison
# ============================================================

def numeric_value(value):
    if is_missing(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    return None


def value_correct(reference_value, extracted_value):

    if is_missing(reference_value) and is_missing(extracted_value):
        return True

    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if reference_number is not None and extracted_number is not None:
        return math.isclose(
            reference_number,
            extracted_number,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

    # Do not coerce text such as one-quarter, one-third, codes,
    # dates, phone numbers or textual labels into numbers.
    if isinstance(reference_value, str) and isinstance(extracted_value, str):
        return (
            normalise_text(reference_value)
            == normalise_text(extracted_value)
        )

    return False


In [14]:
# ============================================================
# 12. Prepare comparison copies and matching blocks
# ============================================================

reference_comparison_df = reference_df.copy()
extracted_comparison_df = pd.DataFrame(
    extracted_records
).copy()

reference_comparison_df["_reference_index"] = np.arange(
    len(reference_comparison_df)
)

extracted_comparison_df["_extraction_index"] = np.arange(
    len(extracted_comparison_df)
)

for frame in [
    reference_comparison_df,
    extracted_comparison_df,
]:
    frame["_block_category"] = frame["Category"].map(
        identity_text
    )

    frame["_block_source_location"] = frame[
        "Source Location"
    ].map(identity_text)


reference_block_counts = (
    reference_comparison_df[
        ["_block_category", "_block_source_location"]
    ]
    .value_counts()
    .to_dict()
)

extracted_block_counts = (
    extracted_comparison_df[
        ["_block_category", "_block_source_location"]
    ]
    .value_counts()
    .to_dict()
)

print("Reference matching blocks:", len(reference_block_counts))
print("Extraction matching blocks:", len(extracted_block_counts))


Reference matching blocks: 9
Extraction matching blocks: 9


In [15]:
# ============================================================
# 13. Identity-only matching score
# ============================================================

def matching_score(reference_row, extracted_row):

    topic_score = text_similarity(
        reference_row["Topic"],
        extracted_row["Topic"],
    )

    description_score = text_similarity(
        reference_row["Description"],
        extracted_row["Description"],
    )

    period_score = text_similarity(
        reference_row["Reporting Period"],
        extracted_row["Reporting Period"],
    )

    total_score = (
        MATCHING_WEIGHTS["topic"] * topic_score
        + MATCHING_WEIGHTS["description"] * description_score
        + MATCHING_WEIGHTS["reporting_period"] * period_score
    )

    return {
        "total": total_score,
        "topic": topic_score,
        "description": description_score,
        "period": period_score,
    }


print(
    "Alignment uses Category + Source Location blocking, then "
    "Topic + Description + Reporting Period identity evidence."
)
print("Value, Unit and Qualifier are excluded from alignment.")


Alignment uses Category + Source Location blocking, then Topic + Description + Reporting Period identity evidence.
Value, Unit and Qualifier are excluded from alignment.


In [16]:
# ============================================================
# 14. One-to-one Hungarian record alignment
# ============================================================

matched_pairs = []
matched_reference_indices = set()
matched_extraction_indices = set()

reference_blocks = reference_comparison_df.groupby(
    ["_block_category", "_block_source_location"],
    dropna=False,
)

for block_key, reference_block in reference_blocks:

    category_key, source_key = block_key

    extracted_block = extracted_comparison_df.loc[
        (
            extracted_comparison_df["_block_category"]
            == category_key
        )
        & (
            extracted_comparison_df["_block_source_location"]
            == source_key
        )
    ]

    if extracted_block.empty:
        continue

    reference_rows = list(reference_block.iterrows())
    extracted_rows = list(extracted_block.iterrows())

    score_matrix = np.zeros(
        (len(reference_rows), len(extracted_rows)),
        dtype=float,
    )

    component_scores = {}

    for i, (_, reference_row) in enumerate(reference_rows):
        for j, (_, extracted_row) in enumerate(extracted_rows):
            scores = matching_score(
                reference_row,
                extracted_row,
            )

            score_matrix[i, j] = scores["total"]
            component_scores[(i, j)] = scores

    row_indices, column_indices = linear_sum_assignment(
        -score_matrix
    )

    for row_i, column_j in zip(
        row_indices,
        column_indices,
    ):
        score = float(score_matrix[row_i, column_j])

        if score < MATCH_SCORE_THRESHOLD:
            continue

        reference_row = reference_rows[row_i][1]
        extracted_row = extracted_rows[column_j][1]
        scores = component_scores[(row_i, column_j)]

        reference_index = int(
            reference_row["_reference_index"]
        )

        extraction_index = int(
            extracted_row["_extraction_index"]
        )

        matched_pairs.append({
            "reference_index": reference_index,
            "extraction_index": extraction_index,
            "matching_score": score,
            "topic_matching_score": scores["topic"],
            "description_matching_score": scores["description"],
            "period_matching_score": scores["period"],
        })

        matched_reference_indices.add(reference_index)
        matched_extraction_indices.add(extraction_index)


matched_pairs = sorted(
    matched_pairs,
    key=lambda item: item["reference_index"],
)

print("Aligned records:", len(matched_pairs))


Aligned records: 47


In [17]:
# ============================================================
# 15. Missing and unsupported/unmatched record tables
# ============================================================

all_reference_indices = set(
    reference_comparison_df["_reference_index"].astype(int)
)

all_extraction_indices = set(
    extracted_comparison_df["_extraction_index"].astype(int)
)

missing_reference_indices = sorted(
    all_reference_indices - matched_reference_indices
)

unsupported_extraction_indices = sorted(
    all_extraction_indices - matched_extraction_indices
)

missing_records_df = (
    reference_comparison_df.loc[
        reference_comparison_df["_reference_index"].isin(
            missing_reference_indices
        ),
        FIELDS + ["_reference_index"],
    ]
    .rename(columns={"_reference_index": "Reference Index"})
    .reset_index(drop=True)
)

unsupported_records_df = (
    extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"].isin(
            unsupported_extraction_indices
        ),
        FIELDS + ["_extraction_index"],
    ]
    .rename(columns={"_extraction_index": "Extraction Index"})
    .reset_index(drop=True)
)

print("Missing reference records:", len(missing_records_df))
print(
    "Unsupported/unmatched extracted records:",
    len(unsupported_records_df),
)

# Automated unmatched records are not labelled hallucinations here.
# Source-grounded adjudication can be performed separately if needed.


Missing reference records: 2
Unsupported/unmatched extracted records: 2


In [18]:
# ============================================================
# 16. Field-level comparison of aligned records
# ============================================================

comparison_rows = []

for pair in matched_pairs:

    reference_row = reference_comparison_df.loc[
        reference_comparison_df["_reference_index"]
        == pair["reference_index"]
    ].iloc[0]

    extracted_row = extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"]
        == pair["extraction_index"]
    ].iloc[0]

    field_matches = {
        "Category": exact_normalised_text_match(
            reference_row["Category"],
            extracted_row["Category"],
        ),

        "Topic": topic_correct(
            reference_row["Topic"],
            extracted_row["Topic"],
        ),

        "Description": exact_normalised_text_match(
            reference_row["Description"],
            extracted_row["Description"],
        ),

        "Value": value_correct(
            reference_row["Value"],
            extracted_row["Value"],
        ),

        "Unit": unit_correct(
            reference_row["Unit"],
            extracted_row["Unit"],
        ),

        "Qualifier": qualifier_correct(
            reference_row["Qualifier"],
            extracted_row["Qualifier"],
        ),

        "Reporting Period": period_correct(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"],
        ),

        "Source Location": exact_normalised_text_match(
            reference_row["Source Location"],
            extracted_row["Source Location"],
        ),
    }

    all_mismatched_fields = [
        field
        for field in FIELDS
        if not field_matches[field]
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    fully_correct = (
        len(primary_mismatched_fields) == 0
    )

    output_row = {
        "Reference Index": pair["reference_index"],
        "Extraction Index": pair["extraction_index"],
        "Category": reference_row["Category"],
        "Matching Score": pair["matching_score"],
        "Topic Matching Score": pair["topic_matching_score"],
        "Description Matching Score":
            pair["description_matching_score"],
        "Reporting Period Matching Score":
            pair["period_matching_score"],
        "Description Lexical Similarity":
            text_similarity(
                reference_row["Description"],
                extracted_row["Description"],
            ),
        "Fully Correct": bool(fully_correct),
        "all_mismatched_fields":
            ", ".join(all_mismatched_fields),
        "primary_mismatched_fields":
            ", ".join(primary_mismatched_fields),
    }

    for field in FIELDS:
        output_row[f"Reference {field}"] = reference_row[field]
        output_row[f"Extracted {field}"] = extracted_row[field]
        output_row[f"{field} Match"] = bool(
            field_matches[field]
        )

    comparison_rows.append(output_row)


comparison_df = pd.DataFrame(comparison_rows)

print("Compared aligned records:", len(comparison_df))

if not comparison_df.empty:
    print(
        "Fully correct primary records:",
        int(comparison_df["Fully Correct"].sum()),
    )

display(comparison_df.head(10))


Compared aligned records: 47
Fully correct primary records: 14


,Reference Index,Extraction Index,Category,Matching Score,Topic Matching Score,Description Matching Score,Reporting Period Matching Score,Description Lexical Similarity,Fully Correct,all_mismatched_fields,...,Unit Match,Reference Qualifier,Extracted Qualifier,Qualifier Match,Reference Reporting Period,Extracted Reporting Period,Reporting Period Match,Reference Source Location,Extracted Source Location,Source Location Match
0,0,0,Project metadata,0.868182,1.0,0.472727,1.0,0.472727,True,Description,...,True,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True
1,1,1,Project metadata,1.000000,1.0,1.000000,1.0,1.000000,False,"Description, Value",...,True,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True
2,2,2,Project metadata,0.879032,1.0,0.516129,1.0,0.516129,True,Description,...,True,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True
3,3,3,Project metadata,0.875000,1.0,0.500000,1.0,0.500000,True,Description,...,True,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True
4,4,4,Project metadata,0.941489,1.0,0.765957,1.0,0.765957,True,Description,...,True,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True
5,5,5,Project metadata,0.884615,1.0,0.538462,1.0,0.538462,True,Description,...,True,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True
6,6,6,Project metadata,1.000000,1.0,1.000000,1.0,1.000000,True,Description,...,True,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True
7,7,7,Project metadata,0.873377,1.0,0.493506,1.0,0.493506,False,"Description, Value",...,True,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True
8,8,8,Project metadata,1.000000,1.0,1.000000,1.0,1.000000,False,"Description, Unit",...,False,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True
9,9,9,Project metadata,1.000000,1.0,1.000000,1.0,1.000000,False,"Description, Unit",...,False,None,None,True,None,None,True,DOC page 1 — Header,DOC page 1 — Header,True


In [19]:
# ============================================================
# 17. Split fully correct and discrepant aligned records
# ============================================================

if comparison_df.empty:
    fully_correct_records_df = comparison_df.copy()
    discrepant_records_df = comparison_df.copy()
else:
    fully_correct_records_df = comparison_df.loc[
        comparison_df["Fully Correct"]
    ].copy()

    discrepant_records_df = comparison_df.loc[
        ~comparison_df["Fully Correct"]
    ].copy()

print("Fully correct aligned records:", len(fully_correct_records_df))
print("Discrepant aligned records:", len(discrepant_records_df))

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Category",
                "primary_mismatched_fields",
            ]
        ].reset_index(drop=True)
    )


Fully correct aligned records: 14
Discrepant aligned records: 33


,Reference Index,Extraction Index,Category,primary_mismatched_fields
0,1,1,Project metadata,Value
1,7,7,Project metadata,Value
2,8,8,Project metadata,Unit
3,9,9,Project metadata,Unit
4,10,10,Project metadata,Unit
5,11,11,Project metadata,Unit
6,12,12,Project metadata,Unit
7,13,13,Development issue,Topic
8,14,14,Development issue,Unit
9,16,16,Development issue,"Topic, Unit"


In [20]:
# ============================================================
# 18. Field-level validation and error summary
# ============================================================

field_rows = []

for field in FIELDS:

    if comparison_df.empty:
        correct_count = 0
        evaluated_count = 0
        accuracy = None
    else:
        evaluated_count = len(comparison_df)
        correct_count = int(
            comparison_df[f"{field} Match"].sum()
        )

        accuracy = (
            correct_count / evaluated_count
            if evaluated_count > 0
            else None
        )

    role = (
        "diagnostic"
        if field == DESCRIPTION_DIAGNOSTIC_FIELD
        else "primary"
    )

    field_rows.append({
        "Field": field,
        "Role": role,
        "Aligned Records": evaluated_count,
        "Correct Records": correct_count,
        "Incorrect Records":
            evaluated_count - correct_count,
        "Accuracy": accuracy,
    })


field_validation_df = pd.DataFrame(field_rows)

field_error_summary_df = field_validation_df.loc[
    field_validation_df["Incorrect Records"] > 0
].copy()

display(field_validation_df)


,Field,Role,Aligned Records,Correct Records,Incorrect Records,Accuracy
0,Category,primary,47,47,0,1.000000
1,Topic,primary,47,34,13,0.723404
2,Description,diagnostic,47,0,47,0.000000
3,Value,primary,47,45,2,0.957447
4,Unit,primary,47,27,20,0.574468
5,Qualifier,primary,47,44,3,0.936170
6,Reporting Period,primary,47,47,0,1.000000
7,Source Location,primary,47,47,0,1.000000


In [21]:
# ============================================================
# 19. Common record-level validation metrics
# ============================================================

reference_record_count = len(reference_df)
extracted_record_count = len(extracted_records)
aligned_record_count = len(comparison_df)

fully_correct_record_count = (
    int(comparison_df["Fully Correct"].sum())
    if not comparison_df.empty
    else 0
)

discrepant_record_count = (
    aligned_record_count - fully_correct_record_count
)

missing_record_count = len(missing_records_df)
unsupported_record_count = len(unsupported_records_df)

completeness = (
    aligned_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

missing_rate = (
    missing_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

unsupported_rate = (
    unsupported_record_count / extracted_record_count
    if extracted_record_count > 0
    else 0.0
)

record_precision_exact = (
    fully_correct_record_count / extracted_record_count
    if extracted_record_count > 0
    else 0.0
)

record_recall_exact = (
    fully_correct_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if record_precision_exact + record_recall_exact > 0
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count / aligned_record_count
    if aligned_record_count > 0
    else 0.0
)

primary_field_rows = field_validation_df.loc[
    field_validation_df["Role"] == "primary"
]

primary_correct_count = int(
    primary_field_rows["Correct Records"].sum()
)

primary_total_count = int(
    primary_field_rows["Aligned Records"].sum()
)

overall_primary_field_accuracy = (
    primary_correct_count / primary_total_count
    if primary_total_count > 0
    else None
)

description_row = field_validation_df.loc[
    field_validation_df["Field"] == "Description"
]

description_diagnostic_accuracy = (
    float(description_row.iloc[0]["Accuracy"])
    if (
        not description_row.empty
        and pd.notna(description_row.iloc[0]["Accuracy"])
    )
    else None
)

print("Reference records:", reference_record_count)
print("Extracted records:", extracted_record_count)
print("Aligned records:", aligned_record_count)
print("Fully correct:", fully_correct_record_count)
print("Discrepant:", discrepant_record_count)
print("Missing:", missing_record_count)
print("Unsupported/unmatched:", unsupported_record_count)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Overall primary field accuracy:",
    None
    if overall_primary_field_accuracy is None
    else round(overall_primary_field_accuracy, 4),
)
print(
    "Description diagnostic accuracy:",
    None
    if description_diagnostic_accuracy is None
    else round(description_diagnostic_accuracy, 4),
)


Reference records: 49
Extracted records: 49
Aligned records: 47
Fully correct: 14
Discrepant: 33
Missing: 2
Unsupported/unmatched: 2
Completeness: 0.9592
Exact F1: 0.2857
Overall primary field accuracy: 0.8845
Description diagnostic accuracy: 0.0


In [22]:
# ============================================================
# 20. Category-level metrics
# ============================================================

category_rows = []

for category in EXPECTED_CATEGORY_COUNTS:

    expected_records = int(
        (reference_df["Category"] == category).sum()
    )

    extracted_records_category = sum(
        1
        for record in extracted_records
        if record.get("Category") == category
    )

    category_comparison = (
        comparison_df.loc[
            comparison_df["Category"] == category
        ]
        if not comparison_df.empty
        else comparison_df
    )

    aligned_records_category = len(category_comparison)

    fully_correct_category = (
        int(category_comparison["Fully Correct"].sum())
        if not category_comparison.empty
        else 0
    )

    discrepant_category = (
        aligned_records_category - fully_correct_category
    )

    category_completeness = (
        aligned_records_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_precision = (
        fully_correct_category / extracted_records_category
        if extracted_records_category > 0
        else 0.0
    )

    category_recall = (
        fully_correct_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_f1 = (
        2 * category_precision * category_recall
        / (category_precision + category_recall)
        if category_precision + category_recall > 0
        else 0.0
    )

    category_rows.append({
        "Category": category,
        "Expected Records": expected_records,
        "Extracted Records": extracted_records_category,
        "Aligned Records": aligned_records_category,
        "Fully Correct Records": fully_correct_category,
        "Discrepant Records": discrepant_category,
        "Completeness": category_completeness,
        "Record Precision Exact": category_precision,
        "Record Recall Exact": category_recall,
        "Record F1 Exact": category_f1,
    })


category_metrics_df = pd.DataFrame(category_rows)
display(category_metrics_df)


,Category,Expected Records,Extracted Records,Aligned Records,Fully Correct Records,Discrepant Records,Completeness,Record Precision Exact,Record Recall Exact,Record F1 Exact
0,Project metadata,13,13,13,6,7,1.000000,0.461538,0.461538,0.461538
1,Development issue,10,10,10,3,7,1.000000,0.300000,0.300000,0.300000
2,Bank rationale,2,2,2,0,2,1.000000,0.000000,0.000000,0.000000
3,Project objective,3,3,3,0,3,1.000000,0.000000,0.000000,0.000000
4,Project component,3,3,1,0,1,0.333333,0.000000,0.000000,0.000000
5,Safeguard policy,6,6,6,0,6,1.000000,0.000000,0.000000,0.000000
6,Financing,7,7,7,0,7,1.000000,0.000000,0.000000,0.000000
7,Contact information,5,5,5,5,0,1.000000,1.000000,1.000000,1.000000


In [23]:
# ============================================================
# 21. Define schema validity independently from completeness
# ============================================================

schema_validity = all([
    top_level_object_valid,
    document_id_correct,
    branch_correct,
    records_is_list,
    record_schema_valid,
    field_types_valid,
])

schema_diagnostics = {
    "valid_json": True,
    "top_level_object_valid": bool(top_level_object_valid),
    "document_id_correct": bool(document_id_correct),
    "branch_correct": bool(branch_correct),
    "records_is_list": bool(records_is_list),
    "record_schema_valid": bool(record_schema_valid),
    "field_types_valid": bool(field_types_valid),
    "records_with_structure_issues":
        int(len(schema_issues_df)),
    "records_with_type_issues":
        int(len(type_issues_df)),
    "field_order_diagnostic_count":
        int(len(field_order_diagnostics_df)),
    "schema_validity": bool(schema_validity),
}

content_diagnostics = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),
    "reference_category_counts_valid":
        bool(reference_category_counts_valid),
    "extraction_record_count_valid":
        bool(extraction_record_count_valid),
    "extraction_category_counts_valid":
        bool(extraction_category_counts_valid),
    "mandatory_fields_complete":
        bool(mandatory_fields_complete),
}

print("Schema validity:", schema_validity)
print(json.dumps(schema_diagnostics, indent=2))


Schema validity: True
{
  "valid_json": true,
  "top_level_object_valid": true,
  "document_id_correct": true,
  "branch_correct": true,
  "records_is_list": true,
  "record_schema_valid": true,
  "field_types_valid": true,
  "records_with_structure_issues": 0,
  "records_with_type_issues": 0,
  "field_order_diagnostic_count": 0,
  "schema_validity": true
}


In [24]:
# ============================================================
# 22. Preserve Branch C normalisation-integrity diagnostics
# ============================================================
# Stage 2 B->C representation integrity is deliberately separate
# from Stage 4 extraction correctness.

with NORMALISATION_INTEGRITY_PATH.open(
    "r",
    encoding="utf-8"
) as f:
    normalisation_integrity = json.load(f)


if normalisation_integrity.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Normalisation-integrity document_id does not match D8."
    )

if normalisation_integrity.get("branch") != BRANCH:
    raise ValueError(
        "Normalisation-integrity branch does not match Branch C."
    )

if normalisation_integrity.get("parent_branch") != "B":
    raise ValueError(
        "Normalisation-integrity parent_branch does not match Branch B."
    )


representation_integrity = {
    "parent_branch":
        normalisation_integrity.get("parent_branch"),

    "parent_equivalence_passed":
        bool(
            normalisation_integrity.get(
                "parent_equivalence_passed",
                False
            )
        ),

    "normalisation_integrity_passed":
        bool(
            normalisation_integrity.get(
                "normalisation_integrity_passed",
                False
            )
        ),

    "page_sequence_preserved":
        normalisation_integrity.get(
            "page_sequence_preserved"
        ),

    "deterministic_representation_verified":
        normalisation_integrity.get(
            "deterministic_representation_verified"
        ),

    "all_section_markers_preserved":
        normalisation_integrity.get(
            "all_section_markers_preserved"
        ),

    "all_source_markers_preserved":
        normalisation_integrity.get(
            "all_source_markers_preserved"
        ),

    "all_financing_labels_preserved":
        normalisation_integrity.get(
            "all_financing_labels_preserved"
        ),

    "numeric_values_preserved":
        normalisation_integrity.get(
            "numeric_values_preserved"
        ),

    "complete_4_page_representation_retained":
        normalisation_integrity.get(
            "complete_4_page_representation_retained"
        ),

    "source_scope_filtering_applied":
        normalisation_integrity.get(
            "source_scope_filtering_applied"
        ),

    "page_removal_applied":
        normalisation_integrity.get(
            "page_removal_applied"
        ),

    "page_cropping_applied":
        normalisation_integrity.get(
            "page_cropping_applied"
        ),

    "ocr_applied":
        normalisation_integrity.get(
            "ocr_applied"
        ),

    "structural_financing_label_reconstruction_inherited_from_branch_B":
        normalisation_integrity.get(
            "structural_financing_label_reconstruction_inherited_from_branch_B"
        ),

    "financing_label_reconstruction_count":
        normalisation_integrity.get(
            "financing_label_reconstruction_count"
        ),

    "unicode_nfkc_normalisation_applied":
        normalisation_integrity.get(
            "unicode_nfkc_normalisation_applied"
        ),

    "unicode_space_standardisation_applied":
        normalisation_integrity.get(
            "unicode_space_standardisation_applied"
        ),

    "apostrophe_standardisation_applied":
        normalisation_integrity.get(
            "apostrophe_standardisation_applied"
        ),

    "dash_and_minus_standardisation_applied":
        normalisation_integrity.get(
            "dash_and_minus_standardisation_applied"
        ),

    "soft_hyphen_removal_applied":
        normalisation_integrity.get(
            "soft_hyphen_removal_applied"
        ),

    "line_endings_standardised":
        normalisation_integrity.get(
            "line_endings_standardised"
        ),

    "horizontal_whitespace_normalisation_applied":
        normalisation_integrity.get(
            "horizontal_whitespace_normalisation_applied"
        ),

    "paragraph_line_merging_applied":
        normalisation_integrity.get(
            "paragraph_line_merging_applied"
        ),

    "line_break_hyphenation_repair_applied":
        normalisation_integrity.get(
            "line_break_hyphenation_repair_applied"
        ),

    "semantic_harmonisation_applied":
        normalisation_integrity.get(
            "semantic_harmonisation_applied"
        ),

    "semantic_rewriting_applied":
        normalisation_integrity.get(
            "semantic_rewriting_applied"
        ),

    "source_spelling_repair_applied":
        normalisation_integrity.get(
            "source_spelling_repair_applied"
        ),

    "unit_conversion_applied":
        normalisation_integrity.get(
            "unit_conversion_applied"
        ),

    "numeric_calculation_applied":
        normalisation_integrity.get(
            "numeric_calculation_applied"
        ),

    "manual_correction_applied":
        normalisation_integrity.get(
            "manual_correction_applied"
        ),

    "reference_values_used_for_transformation":
        normalisation_integrity.get(
            "reference_values_used_for_transformation"
        ),

    "numeric_token_preservation":
        normalisation_integrity.get(
            "numeric_token_preservation"
        ),

    "section_marker_checks":
        normalisation_integrity.get(
            "section_marker_checks"
        ),

    "source_marker_checks":
        normalisation_integrity.get(
            "source_marker_checks"
        ),

    "financing_label_checks":
        normalisation_integrity.get(
            "financing_label_checks"
        ),
}


print("Branch C representation integrity:")
print(
    json.dumps(
        representation_integrity,
        indent=2,
        ensure_ascii=False
    )
)


Branch C representation integrity:
{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "all_section_markers_preserved": true,
  "all_source_markers_preserved": true,
  "all_financing_labels_preserved": true,
  "numeric_values_preserved": true,
  "complete_4_page_representation_retained": true,
  "source_scope_filtering_applied": false,
  "page_removal_applied": false,
  "page_cropping_applied": false,
  "ocr_applied": false,
  "structural_financing_label_reconstruction_inherited_from_branch_B": true,
  "financing_label_reconstruction_count": 2,
  "unicode_nfkc_normalisation_applied": true,
  "unicode_space_standardisation_applied": true,
  "apostrophe_standardisation_applied": true,
  "dash_and_minus_standardisation_applied": true,
  "soft_hyphen_removal_applied": true,
  "line_endings_standardised": true,
  "horizontal_whitespace_normalisation_appl

In [26]:
# ============================================================
# 23. Create reproducible Branch C validation summary
# ============================================================

# Field accuracy comes from Cell 18.
field_accuracy_dictionary = {
    row["Field"]: (
        None
        if pd.isna(row["Accuracy"])
        else float(row["Accuracy"])
    )
    for _, row in field_validation_df.iterrows()
}


# Category metrics come from Cell 20.
category_metrics_dictionary = {
    row["Category"]: {
        "expected_records":
            int(row["Expected Records"]),

        "extracted_records":
            int(row["Extracted Records"]),

        "aligned_records":
            int(row["Aligned Records"]),

        "fully_correct_records":
            int(row["Fully Correct Records"]),

        "discrepant_records":
            int(row["Discrepant Records"]),

        "completeness":
            float(row["Completeness"]),

        "record_precision_exact":
            float(row["Record Precision Exact"]),

        "record_recall_exact":
            float(row["Record Recall Exact"]),

        "record_f1_exact":
            float(row["Record F1 Exact"]),
    }

    for _, row
    in category_metrics_df.iterrows()
}


VALIDATION_METRICS = {

    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,


    # --------------------------------------------------------
    # Record-level outcomes
    # --------------------------------------------------------

    "reference_records":
        int(reference_record_count),

    "extracted_records":
        int(extracted_record_count),

    "aligned_records":
        int(aligned_record_count),

    "fully_correct_records":
        int(fully_correct_record_count),

    "discrepant_records":
        int(discrepant_record_count),

    "missing_records":
        int(missing_record_count),

    "unsupported_extracted_records":
        int(unsupported_record_count),


    # --------------------------------------------------------
    # Record-level metrics
    # --------------------------------------------------------

    "completeness":
        round(
            float(completeness),
            4
        ),

    "missing_rate":
        round(
            float(missing_rate),
            4
        ),

    "record_precision_exact":
        round(
            float(record_precision_exact),
            4
        ),

    "record_recall_exact":
        round(
            float(record_recall_exact),
            4
        ),

    "record_f1_exact":
        round(
            float(record_f1_exact),
            4
        ),

    "unsupported_rate":
        round(
            float(unsupported_rate),
            4
        ),

    "discrepancy_rate_among_aligned":
        round(
            float(
                discrepancy_rate_among_aligned
            ),
            4
        ),


    # --------------------------------------------------------
    # Field-level metrics
    # --------------------------------------------------------

    "overall_primary_field_accuracy":
        (
            None
            if overall_primary_field_accuracy
            is None
            else round(
                float(
                    overall_primary_field_accuracy
                ),
                4
            )
        ),

    "description_diagnostic_accuracy":
        (
            None
            if description_diagnostic_accuracy
            is None
            else round(
                float(
                    description_diagnostic_accuracy
                ),
                4
            )
        ),

    "field_accuracy_among_aligned":
        field_accuracy_dictionary,

    "description_status":
        (
            "Diagnostic only; excluded from primary record "
            "correctness exactly as frozen in final D8 "
            "Branch A validation."
        ),


    # --------------------------------------------------------
    # Schema and content diagnostics
    # --------------------------------------------------------

    "schema_validity":
        bool(
            schema_validity
        ),

    "schema_diagnostics":
        schema_diagnostics,

    "content_diagnostics":
        content_diagnostics,


    # --------------------------------------------------------
    # Branch C representation integrity
    # --------------------------------------------------------

    "branch_C_representation_integrity":
        representation_integrity,


    # --------------------------------------------------------
    # Frozen matching rules
    # --------------------------------------------------------

    "matching_rules": {

        "blocking_fields":
            BLOCK_FIELDS,

        "one_to_one_assignment":
            "Hungarian linear-sum assignment",

        "matching_score_threshold":
            MATCH_SCORE_THRESHOLD,

        "matching_score_weights":
            MATCHING_WEIGHTS,

        "value_used_for_alignment":
            False,

        "unit_used_for_alignment":
            False,

        "qualifier_used_for_alignment":
            False,

        "description_used_for_alignment":
            True,

        "description_used_for_primary_correctness":
            False,
    },


    # --------------------------------------------------------
    # Frozen comparison rules
    # --------------------------------------------------------

    "comparison_rules": {

        "raw_extraction_modified":
            False,

        "manual_correction_applied":
            False,

        "comparison_normalisation_scope":
            "Comparison copies only",

        "numeric_comparison":
            (
                "Signed numeric equality after "
                "deterministic parsing"
            ),

        "absolute_numeric_value_for_correctness":
            False,

        "absolute_numeric_value_for_alignment":
            False,

        "topic":
            (
                "Normalised exact equality; "
                "no Branch-C-specific aliases added."
            ),

        "unit":
            (
                "Frozen safe notation equivalence only: "
                "percent and USD-million forms."
            ),

        "qualifier":
            "Normalised exact equality.",

        "reporting_period":
            "Normalised exact equality.",

        "source_location":
            "Normalised exact correctness.",

        "description":
            (
                "Diagnostic lexical field; "
                "not a primary correctness field."
            ),

        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,

        "d8_rules_status":
            (
                "Final D8 Branch A alignment/comparison "
                "design reused unchanged."
            ),
    },


    # --------------------------------------------------------
    # Category metrics
    # --------------------------------------------------------

    "category_metrics":
        category_metrics_dictionary,


    # --------------------------------------------------------
    # Methodological note
    # --------------------------------------------------------

    "normalisation_note":
        (
            "Stage 4 comparison normalisation is applied only "
            "to comparison copies using rules frozen in D8 "
            "Branch A. This is distinct from the deterministic "
            "Branch C input normalisation; the preserved Branch C "
            "extraction is not modified."
        ),


    # --------------------------------------------------------
    # Input provenance
    # --------------------------------------------------------

    "input_provenance": {

        "reference_file":
            REFERENCE_PATH.name,

        "reference_sha256":
            REFERENCE_SHA256,

        "parsed_extraction_file":
            EXTRACTION_PATH.name,

        "parsed_extraction_sha256":
            EXTRACTION_SHA256,

        "structure_check_file":
            STRUCTURE_CHECK_PATH.name,

        "structure_check_sha256":
            STRUCTURE_CHECK_SHA256,

        "experiment_metadata_file":
            EXPERIMENT_METADATA_PATH.name,

        "experiment_metadata_sha256":
            EXPERIMENT_METADATA_SHA256,

        "normalisation_integrity_file":
            NORMALISATION_INTEGRITY_PATH.name,

        "normalisation_integrity_sha256":
            NORMALISATION_INTEGRITY_SHA256,

        "experiment_summary_file":
            (
                EXPERIMENT_SUMMARY_PATH.name
                if EXPERIMENT_SUMMARY_PATH
                is not None
                else None
            ),

        "experiment_summary_sha256":
            EXPERIMENT_SUMMARY_SHA256,

        "branch_C_structure_valid":
            bool(
                branch_c_structure_valid
            ),

        "parsed_extraction_hash_matches_metadata":
            bool(
                parsed_extraction_hash_matches_metadata
            ),

        "source_hash_matches_stage_1":
            bool(
                source_hash_matches_stage_1
            ),
    },


    "comparison_rules_frozen_from_branch_A":
        True,

    "validation_timestamp":
        datetime.now().isoformat(),
}


# ============================================================
# One-row tabular summary for CSV export
# ============================================================

validation_summary_df = pd.DataFrame([{

    "Document ID":
        DOCUMENT_ID,

    "Branch":
        BRANCH,

    "Reference Records":
        reference_record_count,

    "Extracted Records":
        extracted_record_count,

    "Aligned Records":
        aligned_record_count,

    "Fully Correct Records":
        fully_correct_record_count,

    "Discrepant Records":
        discrepant_record_count,

    "Missing Records":
        missing_record_count,

    "Unsupported Extracted Records":
        unsupported_record_count,

    "Completeness":
        completeness,

    "Record Precision Exact":
        record_precision_exact,

    "Record Recall Exact":
        record_recall_exact,

    "Record F1 Exact":
        record_f1_exact,

    "Overall Primary Field Accuracy":
        overall_primary_field_accuracy,

    "Description Diagnostic Accuracy":
        description_diagnostic_accuracy,

    "Schema Validity":
        schema_validity,

    # Correct Branch C integrity field.
    "Normalisation Integrity Passed":
        representation_integrity[
            "normalisation_integrity_passed"
        ],
}])


print(
    json.dumps(
        VALIDATION_METRICS,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D8",
  "document_name": "World Bank — Bhutan - Land Management Project — Project Information Document (PID), Concept Stage",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised page-aware structural Markdown",
  "reference_records": 49,
  "extracted_records": 49,
  "aligned_records": 47,
  "fully_correct_records": 14,
  "discrepant_records": 33,
  "missing_records": 2,
  "unsupported_extracted_records": 2,
  "completeness": 0.9592,
  "missing_rate": 0.0408,
  "record_precision_exact": 0.2857,
  "record_recall_exact": 0.2857,
  "record_f1_exact": 0.2857,
  "unsupported_rate": 0.0408,
  "discrepancy_rate_among_aligned": 0.7021,
  "overall_primary_field_accuracy": 0.8845,
  "description_diagnostic_accuracy": 0.0,
  "field_accuracy_among_aligned": {
    "Category": 1.0,
    "Topic": 0.723404255319149,
    "Description": 0.0,
    "Value": 0.9574468085106383,
    "Unit": 0.574468085106383,
    "

In [27]:
# ============================================================
# 24. Validation metadata and conclusion
# ============================================================

validation_status = (
    "Completed without discrepancies"
    if (
        fully_correct_record_count == reference_record_count
        and missing_record_count == 0
        and unsupported_record_count == 0
        and schema_validity
    )
    else "Completed with discrepancies"
)

VALIDATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "reference_file": REFERENCE_PATH.name,
    "reference_file_sha256": REFERENCE_SHA256,
    "extraction_file": EXTRACTION_PATH.name,
    "extraction_file_sha256": EXTRACTION_SHA256,
    "structure_check_file": STRUCTURE_CHECK_PATH.name,
    "structure_check_file_sha256": STRUCTURE_CHECK_SHA256,
    "experiment_metadata_file": EXPERIMENT_METADATA_PATH.name,
    "experiment_metadata_file_sha256": EXPERIMENT_METADATA_SHA256,
    "normalisation_integrity_file": NORMALISATION_INTEGRITY_PATH.name,
    "normalisation_integrity_file_sha256": NORMALISATION_INTEGRITY_SHA256,
    "validation_type": (
        "Deterministic comparison against the fixed 49-record "
        "D8 Stage 1 reference dataset"
    ),
    "raw_extraction_modified": False,
    "manual_correction_applied": False,
    "schema_errors_preserved": True,
    "comparison_normalisation_scope": "Comparison copies only",
    "matching_outcome_values_used": False,
    "automatic_unmatched_label":
        "unsupported/unmatched; not automatically hallucinated",
    "description_primary_correctness_field": False,
    "comparison_rules_frozen_from_branch_A": True,
    "created_at": datetime.now().isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
}

VALIDATION_CONCLUSION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "validation_status": validation_status,
    "reference_records": int(reference_record_count),
    "extracted_records": int(extracted_record_count),
    "aligned_records": int(aligned_record_count),
    "fully_correct_records": int(fully_correct_record_count),
    "discrepant_records": int(discrepant_record_count),
    "missing_records": int(missing_record_count),
    "unsupported_extracted_records": int(unsupported_record_count),
    "completeness": round(float(completeness), 4),
    "record_precision_exact": round(float(record_precision_exact), 4),
    "record_recall_exact": round(float(record_recall_exact), 4),
    "record_f1_exact": round(float(record_f1_exact), 4),
    "overall_primary_field_accuracy": (
        None
        if overall_primary_field_accuracy is None
        else round(float(overall_primary_field_accuracy), 4)
    ),
    "description_diagnostic_accuracy": (
        None
        if description_diagnostic_accuracy is None
        else round(float(description_diagnostic_accuracy), 4)
    ),
    "schema_valid": bool(schema_validity),
    "normalisation_integrity_passed":
        representation_integrity["normalisation_integrity_passed"],
    "comparison_rules_frozen_from_branch_A": True,
}

print(json.dumps(
    VALIDATION_CONCLUSION,
    ensure_ascii=False,
    indent=2
))


{
  "document_id": "D8",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "validation_status": "Completed with discrepancies",
  "reference_records": 49,
  "extracted_records": 49,
  "aligned_records": 47,
  "fully_correct_records": 14,
  "discrepant_records": 33,
  "missing_records": 2,
  "unsupported_extracted_records": 2,
  "completeness": 0.9592,
  "record_precision_exact": 0.2857,
  "record_recall_exact": 0.2857,
  "record_f1_exact": 0.2857,
  "overall_primary_field_accuracy": 0.8845,
  "description_diagnostic_accuracy": 0.0,
  "schema_valid": true,
  "normalisation_integrity_passed": true,
  "comparison_rules_frozen_from_branch_A": true
}


In [28]:
# ============================================================
# 25. Export Validation C outputs
# ============================================================

DETAILED_PATH = OUTPUT_DIR / "D8_branch_C_validation_detailed.csv"
FULLY_CORRECT_PATH = OUTPUT_DIR / "D8_branch_C_fully_correct_records.csv"
DISCREPANT_PATH = OUTPUT_DIR / "D8_branch_C_discrepant_records.csv"
MISSING_PATH = OUTPUT_DIR / "D8_branch_C_missing_records.csv"
UNSUPPORTED_PATH = OUTPUT_DIR / "D8_branch_C_unsupported_records.csv"
SCHEMA_ISSUES_PATH = OUTPUT_DIR / "D8_branch_C_schema_issues.csv"
TYPE_ISSUES_PATH = OUTPUT_DIR / "D8_branch_C_type_issues.csv"
FIELD_VALIDATION_PATH = OUTPUT_DIR / "D8_branch_C_field_validation.csv"
FIELD_ERROR_SUMMARY_PATH = OUTPUT_DIR / "D8_branch_C_field_error_summary.csv"
CATEGORY_METRICS_PATH = OUTPUT_DIR / "D8_branch_C_category_metrics.csv"
VALIDATION_SUMMARY_CSV_PATH = OUTPUT_DIR / "D8_branch_C_validation_summary.csv"
VALIDATION_SUMMARY_JSON_PATH = OUTPUT_DIR / "D8_branch_C_validation_summary.json"
VALIDATION_METADATA_PATH = OUTPUT_DIR / "D8_branch_C_validation_metadata.json"
VALIDATION_CONCLUSION_PATH = OUTPUT_DIR / "D8_branch_C_validation_conclusion.json"

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_PATH,
    index=False,
    encoding="utf-8-sig"
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig"
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig"
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig"
)

schema_issues_df.to_csv(
    SCHEMA_ISSUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

type_issues_df.to_csv(
    TYPE_ISSUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_validation_df.to_csv(
    FIELD_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_error_summary_df.to_csv(
    FIELD_ERROR_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

validation_summary_df.to_csv(
    VALIDATION_SUMMARY_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

VALIDATION_SUMMARY_JSON_PATH.write_text(
    json.dumps(
        VALIDATION_METRICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
)

VALIDATION_METADATA_PATH.write_text(
    json.dumps(
        VALIDATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
)

VALIDATION_CONCLUSION_PATH.write_text(
    json.dumps(
        VALIDATION_CONCLUSION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
)

print("Validation C artefacts saved.")


Validation C artefacts saved.


In [29]:
# ============================================================
# 26. Final consistency checks
# ============================================================

# Fixed Stage 1 reference integrity
if not reference_schema_exact:
    raise AssertionError(
        "Reference schema validation failed."
    )

if not reference_record_count_valid:
    raise AssertionError(
        "Reference record-count validation failed."
    )

if not reference_category_counts_valid:
    raise AssertionError(
        "Reference category-count validation failed."
    )

# Branch C extraction structural/type validity
if not record_schema_valid:
    raise AssertionError(
        "Branch C extracted-record schema validation failed."
    )

if not field_types_valid:
    raise AssertionError(
        "The extracted comparison fields contain invalid data types."
    )

# Stage 2 Branch C normalisation integrity
if not representation_integrity["normalisation_integrity_passed"]:
    raise AssertionError(
        "Branch C normalisation-integrity checks did not pass."
    )

# Stage 4 reference accounting
if (
    aligned_record_count + missing_record_count
    != reference_record_count
):
    raise AssertionError(
        "Reference-record accounting is inconsistent."
    )

# Stage 4 extraction accounting
if (
    aligned_record_count + unsupported_record_count
    != extracted_record_count
):
    raise AssertionError(
        "Extraction-record accounting is inconsistent."
    )

# Stage 4 aligned-record correctness accounting
if (
    fully_correct_record_count + discrepant_record_count
    != aligned_record_count
):
    raise AssertionError(
        "Aligned-record correctness accounting is inconsistent."
    )

print("Validation status:", validation_status)

print("\nReference integrity:")
print("Reference schema exact:", reference_schema_exact)
print("Reference record count valid:", reference_record_count_valid)
print(
    "Reference category counts valid:",
    reference_category_counts_valid
)

print("\nExtraction/schema integrity:")
print("Record schema valid:", record_schema_valid)
print("Field types valid:", field_types_valid)
print("Schema valid:", schema_validity)

print("\nBranch C representation integrity:")
print(
    "Normalisation integrity passed:",
    representation_integrity["normalisation_integrity_passed"]
)

print("\nRecord accounting:")
print("Reference records:", reference_record_count)
print("Extracted records:", extracted_record_count)
print("Aligned records:", aligned_record_count)
print("Missing records:", missing_record_count)
print(
    "Unsupported/unmatched records:",
    unsupported_record_count
)
print("Fully correct records:", fully_correct_record_count)
print("Discrepant records:", discrepant_record_count)

print("\nPerformance:")
print("Completeness:", completeness)
print("Exact precision:", record_precision_exact)
print("Exact recall:", record_recall_exact)
print("Exact F1:", record_f1_exact)
print(
    "Overall primary field accuracy:",
    overall_primary_field_accuracy
)
print(
    "Description diagnostic accuracy:",
    description_diagnostic_accuracy
)

print("\nD8 Validation C completed successfully.")


Validation status: Completed with discrepancies

Reference integrity:
Reference schema exact: True
Reference record count valid: True
Reference category counts valid: True

Extraction/schema integrity:
Record schema valid: True
Field types valid: True
Schema valid: True

Branch C representation integrity:
Normalisation integrity passed: True

Record accounting:
Reference records: 49
Extracted records: 49
Aligned records: 47
Missing records: 2
Unsupported/unmatched records: 2
Fully correct records: 14
Discrepant records: 33

Performance:
Completeness: 0.9591836734693877
Exact precision: 0.2857142857142857
Exact recall: 0.2857142857142857
Exact F1: 0.2857142857142857
Overall primary field accuracy: 0.8844984802431611
Description diagnostic accuracy: 0.0

D8 Validation C completed successfully.


In [30]:
# ============================================================
# 27. Download generated Validation C outputs
# ============================================================

GENERATED_OUTPUTS = [
    DETAILED_PATH,
    FULLY_CORRECT_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    SCHEMA_ISSUES_PATH,
    TYPE_ISSUES_PATH,
    FIELD_VALIDATION_PATH,
    FIELD_ERROR_SUMMARY_PATH,
    CATEGORY_METRICS_PATH,
    VALIDATION_SUMMARY_CSV_PATH,
    VALIDATION_SUMMARY_JSON_PATH,
    VALIDATION_METADATA_PATH,
    VALIDATION_CONCLUSION_PATH,
]

print("Generated D8 Validation C files:\n")

for output_path in GENERATED_OUTPUTS:
    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

for output_path in GENERATED_OUTPUTS:
    if output_path.exists():
        files.download(output_path)


Generated D8 Validation C files:

- D8_branch_C_validation_detailed.csv | exists: True
- D8_branch_C_fully_correct_records.csv | exists: True
- D8_branch_C_discrepant_records.csv | exists: True
- D8_branch_C_missing_records.csv | exists: True
- D8_branch_C_unsupported_records.csv | exists: True
- D8_branch_C_schema_issues.csv | exists: True
- D8_branch_C_type_issues.csv | exists: True
- D8_branch_C_field_validation.csv | exists: True
- D8_branch_C_field_error_summary.csv | exists: True
- D8_branch_C_category_metrics.csv | exists: True
- D8_branch_C_validation_summary.csv | exists: True
- D8_branch_C_validation_summary.json | exists: True
- D8_branch_C_validation_metadata.json | exists: True
- D8_branch_C_validation_conclusion.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>